In [ ]:
"""
This script is responsible for data acquisition by filtering and processing data 
from an SQL database using predefined queries. The retrieved data is then used 
for further calculations in subsequent processing steps.
"""

import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime

# Third-party libraries
import requests  # For making HTTP requests
import pandas as pd  # For data manipulation
import sqlalchemy  # For database interaction
from sqlalchemy.engine import URL, create_engine
from shapely import wkb  # For geospatial data processing

# This is used to cache function return values (SQL query text → return value)
from joblib import Memory 

# Custom code and numerical/visualization tools
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For data visualization
import seaborn as sns  # For Seaborn-based plots
import plotly.express as px  # For interactive visualizations using Plotly
import geopandas as gpd  # For geospatial data analysis
from matplotlib.dates import HourLocator, DateFormatter
from utilities import run_sql  # Custom SQL execution utility

# Determine module path (handles Jupyter Notebook and Python script cases)
try:
    MODULE_PATH = Path(__file__).resolve().parent.parent  # Standard for scripts
    SCRIPT_PATH = Path(__file__).resolve()
except NameError:
    MODULE_PATH = Path.cwd().parent  # Jupyter Notebook case
    SCRIPT_PATH = MODULE_PATH / "data_acquisition.ipynb"  # Default fallback for Jupyter

if MODULE_PATH.as_posix() not in sys.path:
    sys.path.append(MODULE_PATH.as_posix())

# Change working directory to project root dynamically
PROJECT_ROOT = MODULE_PATH  # Assuming repo root is two levels up
os.chdir(PROJECT_ROOT)

# Setup caching for SQL queries (cache directory: `.cache/`)
CACHE_DIR = PROJECT_ROOT / ".cache"
CACHE_DIR.mkdir(exist_ok=True)  # Ensure cache directory exists
memory = Memory(CACHE_DIR, verbose=0)


@memory.cache
def cached_run_sql(query: str, version: int = 1):  # Increment version when query changes
    """
    Executes an SQL query and caches the result.

    :param query: SQL query string
    :param version: Version number to invalidate cache if needed
    :return: Query result (cached)
    """
    return run_sql(query)


In [ ]:
# # Ensure old cached results are removed
# memory.clear(warn=False)

# Trips are stored in anon_master (this is just a join of (materialized) views anon_*)
df_trips = run_sql('trips', index_col='track_id')

# TODO update to add freight forwarder number based on vehicle_id:
df_trips['freight_forwarder'] = df_trips['vehicle_id'].apply(
    lambda x: 1 if x <= 10 else (2 if x <= 20 else (3 if x <= 30 else 4))
)

# Define output path using relative paths
output_path = PROJECT_ROOT / "input" / "tracks.csv"
df_trips.to_csv(output_path)

df_trips


In [ ]:
# fleet information (especially fleet id)
df_fleet = run_sql('fleet', index_col='vehicle_id')
# Define output path using relative paths
output_path = PROJECT_ROOT / "input" / "fleet.csv"
df_fleet.to_csv(output_path)
df_fleet.head()

In [ ]:
# track_id map information

df_track_ids = run_sql('track_ids')
assert all(df_track_ids.index == df_track_ids['track_id_new']), 'Index does not match track_id_new'
df_track_ids.set_index('track_id_new', inplace=True)

output_path = PROJECT_ROOT / "input" / "track_ids.csv"
df_track_ids.to_csv(output_path)
df_track_ids.head()

In [ ]:
 # trips are stored in anon_master (this is just a join of (materialized) views anon_*
df_nuts3_trips = pd.read_sql('''select t.fid, n, ST_Transform(nm.geom, 3857) as geom from 
(
select substring(anz.fid,0,6) as fid, sum(count) as n from nefton_rio_telemetry.anon_nuts_zones anz
group by substring(anz.fid,0,6)
order by n desc
) t
join administrative_boundaries.nuts_2021_10m nm on t.fid  = nm.fid
where n > 0
order by n desc''', con=engine, index_col='fid')
# TODO Why is the db nefton_rio_telemetry used here? Shouldn't it be spirite? Should I do the same?
# sql script for the creation of nefton_rio_telemetry is in old repo on LRZ syncandshare

df_nuts0 = pd.read_sql('''select gid, fid, ST_Transform(nm.geom, 3857) as geom from 
administrative_boundaries.nuts_2021_10m nm where levl_code = 0''', con=engine, index_col='gid')

df_nuts1 = pd.read_sql('''select gid, fid, ST_Transform(nm.geom, 3857) as geom from 
administrative_boundaries.nuts_2021_10m nm where levl_code = 1''', con=engine, index_col='gid')

gdf_nuts3_trips = gpd.GeoDataFrame(df_nuts3_trips[['n']], geometry=df_nuts3_trips.geom.apply(lambda x: wkb.loads(x, hex=True)))

gdf_nuts0 = gpd.GeoDataFrame(df_nuts0[['fid']], geometry=df_nuts0.geom.apply(lambda x: wkb.loads(x, hex=True)))
gdf_nuts1 = gpd.GeoDataFrame(df_nuts1[['fid']], geometry=df_nuts1.geom.apply(lambda x: wkb.loads(x, hex=True)))


#TODO: Für Heatmap nutzen und Gedanken dazu machen, wie man das anonymisieren könnte

# locations - not to be published only for the heat map!
df_start_locations = pd.read_sql('SELECT track_id, start_location from spirite.anon_layer_base', con=engine, index_col='track_id')
df_start_locations['start_location'] = df_start_locations.start_location.apply(lambda x: wkb.loads(x, hex=True))
df_start_locations.to_pickle('input/df_start_locations.pkl')
df_start_locations.head()